# The Importance of Loss Functions — Easy Exercises

This notebook illustrates how the **choice of loss function** affects the fitted model.

We study two settings:

1. **Linear regression**
   - without outliers
   - with strong outliers
2. **Binary classification**
   - logistic loss
   - hinge loss

Some simple parts of the code have been removed as **short exercises**.
Answer the questions and complete the marked lines.

In [ ]:
import numpy as np\nimport matplotlib.pyplot as plt\n\nfrom sklearn.linear_model import LinearRegression, LogisticRegression\nfrom sklearn.svm import LinearSVC\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom scipy.optimize import minimize\n\nnp.random.seed(2)

## 1. Regression without outliers

In [ ]:
n_clean = 30\n\nX_clean = np.linspace(0, 10, n_clean)\n\n# Exercise: complete the data-generating model\n# y_clean = 2 * X_clean + 1 + Gaussian noise\ny_clean = 2 * X_clean + 1 + np.random.normal(0, 0.8, n_clean)

### Question\n\nWhat is the underlying linear model used to generate the data?\n\nWhat role does the noise term play?

In [ ]:
plt.figure(figsize=(7,5))\nplt.scatter(X_clean, y_clean, s=55, alpha=0.85)\nplt.xlabel("x")\nplt.ylabel("y")\nplt.title("Regression data without outliers")\nplt.grid(True, alpha=0.3)\nplt.show()

## Loss functions in regression\n\nIn supervised learning, a model is trained by minimizing a **loss function** that measures the discrepancy between predictions and observed data.\n\nLet\n\nyᵢ = true value  \nŷᵢ = predicted value  \n\nThe **residual** is\n\neᵢ = yᵢ − ŷᵢ\n\nA loss function assigns a penalty to each residual. The total objective minimized during training is typically the **sum (or average) of these penalties**.

In [ ]:
e = np.linspace(-6, 6, 500)\ndelta = 1.0\n\n# Exercise: complete these two losses\nmse = e**2\nmae = np.abs(e)\n\n# Huber\nhuber = np.where(np.abs(e) <= delta,\n                 0.5 * e**2,\n                 delta * (np.abs(e) - 0.5 * delta))\n\n# Exercise: implement log-cosh\nlog_cosh = np.log(np.cosh(e))

In [ ]:
plt.figure(figsize=(8,5))\nplt.plot(e, mse, label="MSE", linewidth=2)\nplt.plot(e, mae, label="MAE", linewidth=2)\nplt.plot(e, huber, label="Huber", linewidth=2)\nplt.plot(e, log_cosh, label="Log-Cosh", linewidth=2)\nplt.xlabel("Residual e")\nplt.ylabel("Loss")\nplt.title("Common regression losses")\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.show()

### Question\n\nWhich loss grows fastest for large residuals?\n\nWhich losses seem more robust to outliers just by looking at the plot?

## Derivatives with respect to the residual\n\nWhen optimizing model parameters using gradient-based methods, we compute derivatives of the loss with respect to the parameters.\n\nHowever, these derivatives depend on the derivative of the loss with respect to the **residual**.\n\nIf\n\ne = y − ŷ\n\nthen by the chain rule\n\n∂L/∂θ = (∂L/∂e) (∂e/∂θ)\n\nTherefore, understanding the derivative of the loss as a function of the residual helps us understand how optimization algorithms react to errors of different magnitudes.

### A note on MAE\n\nThe Mean Absolute Error is\n\nL(e) = |e|\n\nThis function is **not differentiable at e = 0**.\n\nIn optimization, this issue is handled using the concept of a **subgradient**. At e = 0, any value in the interval [-1,1] is a valid subgradient.\n\nFor visualization purposes, we use the convention\n\nsign(0) = 0

In [ ]:
# Exercise: complete these derivatives / subgradients\nmse_grad = 2 * e\nmae_grad = np.sign(e)\n\nhuber_grad = np.where(np.abs(e) <= delta,\n                      e,\n                      delta * np.sign(e))\n\nlog_cosh_grad = np.tanh(e)

In [ ]:
plt.figure(figsize=(8,5))\nplt.plot(e, mse_grad, label="MSE derivative", linewidth=2)\nplt.plot(e, mae_grad, label="MAE subgradient", linewidth=2)\nplt.plot(e, huber_grad, label="Huber derivative", linewidth=2)\nplt.plot(e, log_cosh_grad, label="Log-Cosh derivative", linewidth=2)\nplt.xlabel("Residual e")\nplt.ylabel("Derivative")\nplt.title("Derivatives with respect to the residual")\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.show()

### Question\n\nWhy does the behaviour of the derivative help explain robustness to outliers?\n\nWhich derivative keeps growing as |e| becomes large?

## Fitting regression lines with different losses

In [ ]:
def predict_line(theta, x):\n    a, b = theta\n    return a * x + b\n\ndef objective_mse(theta, x, y):\n    r = y - predict_line(theta, x)\n    return np.mean(r**2)\n\ndef objective_mae(theta, x, y):\n    r = y - predict_line(theta, x)\n    return np.mean(np.abs(r))\n\ndef objective_huber(theta, x, y, delta=1.0):\n    r = y - predict_line(theta, x)\n    loss = np.where(np.abs(r) <= delta,\n                    0.5 * r**2,\n                    delta * (np.abs(r) - 0.5 * delta))\n    return np.mean(loss)\n\ndef objective_log_cosh(theta, x, y):\n    r = y - predict_line(theta, x)\n    return np.mean(np.log(np.cosh(r)))\n\ndef grad_log_cosh(theta, x, y):\n    a, b = theta\n    r = y - (a * x + b)\n    t = np.tanh(r)\n    da = -np.mean(t * x)\n    db = -np.mean(t)\n    return np.array([da, db])

In [ ]:
theta0 = np.array([0.0, 0.0])\n\nres_mse_clean = minimize(objective_mse, theta0, args=(X_clean, y_clean))\nres_mae_clean = minimize(objective_mae, theta0, args=(X_clean, y_clean), method="Powell")\nres_huber_clean = minimize(objective_huber, theta0, args=(X_clean, y_clean))\nres_logcosh_clean = minimize(objective_log_cosh, theta0, args=(X_clean, y_clean), jac=grad_log_cosh)\n\ntheta_mse_clean = res_mse_clean.x\ntheta_mae_clean = res_mae_clean.x\ntheta_huber_clean = res_huber_clean.x\ntheta_logcosh_clean = res_logcosh_clean.x

In [ ]:
x_plot = np.linspace(0, 10, 200)\n\nplt.figure(figsize=(8,5))\nplt.scatter(X_clean, y_clean, s=55, alpha=0.85, label="Data")\nplt.plot(x_plot, predict_line(theta_mse_clean, x_plot), label="MSE fit", linewidth=2)\nplt.plot(x_plot, predict_line(theta_mae_clean, x_plot), label="MAE fit", linewidth=2)\nplt.plot(x_plot, predict_line(theta_huber_clean, x_plot), label="Huber fit", linewidth=2)\nplt.plot(x_plot, predict_line(theta_logcosh_clean, x_plot), label="Log-Cosh fit", linewidth=2)\nplt.xlabel("x")\nplt.ylabel("y")\nplt.title("Fits without outliers")\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.show()

### Question\n\nWhen the data contain no strong outliers, do the fitted lines differ much?\n\nWhy might all losses give similar answers in this case?

## 2. Regression with outliers

In [ ]:
n = 30\nX = np.linspace(0, 10, n)\ny = 2 * X + 1 + np.random.normal(0, 0.8, n)\n\n# Add strong outliers\nX_out = np.array([2.0, 8.5])\ny_out = np.array([18.0, -3.0])\n\nX_all = np.concatenate([X, X_out])\ny_all = np.concatenate([y, y_out])

In [ ]:
plt.figure(figsize=(7,5))\nplt.scatter(X_all, y_all, s=55, alpha=0.9)\nplt.xlabel("x")\nplt.ylabel("y")\nplt.title("Regression data with outliers")\nplt.grid(True, alpha=0.3)\nplt.show()

## Sensitivity to outliers\n\nDifferent loss functions react very differently to large residuals.\n\n- **Mean Squared Error (MSE)** squares the residuals, so large errors receive a very large penalty.\n- **Huber loss** behaves quadratically for small residuals and linearly for large ones.\n- **Log-Cosh loss** is smooth everywhere. Near zero it behaves approximately like the squared loss, while for large residuals it grows approximately linearly.\n\n> The loss function encodes which errors matter most during training.

In [ ]:
res_mse = minimize(objective_mse, theta0, args=(X_all, y_all))\nres_mae = minimize(objective_mae, theta0, args=(X_all, y_all), method="Powell")\nres_huber = minimize(objective_huber, theta0, args=(X_all, y_all))\nres_logcosh = minimize(objective_log_cosh, theta0, args=(X_all, y_all), jac=grad_log_cosh)\n\ntheta_mse = res_mse.x\ntheta_mae = res_mae.x\ntheta_huber = res_huber.x\ntheta_logcosh = res_logcosh.x

In [ ]:
plt.figure(figsize=(8,5))\nplt.scatter(X_all, y_all, s=55, alpha=0.9, label="Data")\nplt.plot(x_plot, predict_line(theta_mse, x_plot), label="MSE fit", linewidth=2)\nplt.plot(x_plot, predict_line(theta_mae, x_plot), label="MAE fit", linewidth=2)\nplt.plot(x_plot, predict_line(theta_huber, x_plot), label="Huber fit", linewidth=2)\nplt.plot(x_plot, predict_line(theta_logcosh, x_plot), label="Log-Cosh fit", linewidth=2)\nplt.xlabel("x")\nplt.ylabel("y")\nplt.title("Fits with outliers")\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.show()

### Question\n\nWhich fitted line changes the most because of the outliers?\n\nWhich losses appear more resistant to the outliers?

## Optional extension: gradients and curvature\n\nTo understand why some losses are more robust than others, it is useful to compare not only their values, but also their **gradients** and **second derivatives**.\n\n- The **gradient** shows how strongly the loss reacts to an error.\n- The **second derivative** shows the local curvature of the objective.\n\nThese quantities help explain the behaviour of optimization algorithms and the sensitivity of the fitted model to outliers.

In [ ]:
# Exercise: compare gradient magnitude and curvature of common regression losses\n\nmse_hess = 2 * np.ones_like(e)\nmae_hess = np.zeros_like(e)\nhuber_hess = np.where(np.abs(e) <= delta, 1.0, 0.0)\nlogcosh_hess = 1 / np.cosh(e)**2\n\nplt.figure(figsize=(8,5))\nplt.plot(e, np.abs(mse_grad), label="|MSE gradient|", linewidth=2)\nplt.plot(e, np.abs(mae_grad), label="|MAE subgradient|", linewidth=2)\nplt.plot(e, np.abs(huber_grad), label="|Huber gradient|", linewidth=2)\nplt.plot(e, np.abs(log_cosh_grad), label="|Log-Cosh gradient|", linewidth=2)\nplt.axvline(delta, linestyle="--", alpha=0.6)\nplt.axvline(-delta, linestyle="--", alpha=0.6)\nplt.xlabel("Residual e")\nplt.ylabel("Gradient magnitude")\nplt.title("How strongly each loss reacts to error")\nplt.legend()\nplt.grid(True)\nplt.show()\n\nplt.figure(figsize=(8,5))\nplt.plot(e, mse_hess, label="MSE curvature", linewidth=2)\nplt.plot(e, mae_hess, label="MAE curvature", linewidth=2)\nplt.plot(e, huber_hess, label="Huber curvature", linewidth=2)\nplt.plot(e, logcosh_hess, label="Log-Cosh curvature", linewidth=2)\nplt.axvline(delta, linestyle="--", alpha=0.6)\nplt.axvline(-delta, linestyle="--", alpha=0.6)\nplt.xlabel("Residual e")\nplt.ylabel("Second derivative")\nplt.title("Loss curvature")\nplt.legend()\nplt.grid(True)\nplt.show()

### Question\n\nWhy does a bounded gradient make a loss function more robust to outliers?\n\nHow is that reflected in the fitted regression line?

## 3. Loss functions for classification\n\nIn binary classification we typically learn a model that produces a **score**\n\nf(x)\n\nwhose sign determines the predicted class.\n\nMany learning algorithms use the **margin**\n\nm = y f(x)\n\nwhere y ∈ {−1, +1}.\n\nA positive margin means the prediction is correct, while a negative margin corresponds to a misclassification.

In [ ]:
m = np.linspace(-4, 4, 500)\n\n# Exercise: complete the hinge loss\nlogistic_loss = np.log1p(np.exp(-m))\nhinge_loss = np.maximum(0, 1 - m)\n\nlogistic_grad = -1 / (1 + np.exp(m))\nhinge_grad = np.where(m < 1, -1.0, 0.0)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))\n\naxes[0].plot(m, logistic_loss, label="Logistic loss", linewidth=2)\naxes[0].plot(m, hinge_loss, label="Hinge loss", linewidth=2)\naxes[0].set_xlabel("Margin m")\naxes[0].set_ylabel("Loss")\naxes[0].set_title("Classification losses")\naxes[0].legend()\naxes[0].grid(True, alpha=0.3)\n\naxes[1].plot(m, logistic_grad, label="Logistic gradient", linewidth=2)\naxes[1].plot(m, hinge_grad, label="Hinge subgradient", linewidth=2)\naxes[1].set_xlabel("Margin m")\naxes[1].set_ylabel("Derivative")\naxes[1].set_title("Derivatives with respect to the margin")\naxes[1].legend()\naxes[1].grid(True, alpha=0.3)\n\nplt.tight_layout()\nplt.show()

## Logistic loss vs hinge loss\n\nTwo common losses used for linear classification are:\n\n**Logistic loss**\n\nℓ_log(m) = log(1 + e^(−m))\n\n**Hinge loss**\n\nℓ_hinge(m) = max(0, 1 − m)\n\nThese losses behave differently:\n\n- Logistic loss decreases smoothly as the margin increases.\n- Hinge loss becomes zero once the margin is larger than 1.

### Question\n\nWhat happens to the hinge loss once the margin is larger than 1?\n\nWhy does logistic loss keep decreasing even for large positive margins?

In [ ]:
np.random.seed(1)\nn_per_class = 35\n\nX_pos = np.random.multivariate_normal([2, 2], [[0.9, 0.2], [0.2, 0.9]], n_per_class)\nX_neg = np.random.multivariate_normal([-2, -2], [[0.9, -0.2], [-0.2, 0.9]], n_per_class)\n\nX_cls = np.vstack([X_pos, X_neg])\ny_cls = np.hstack([np.ones(n_per_class), -np.ones(n_per_class)])

In [ ]:
plt.figure(figsize=(7,5))\nplt.scatter(X_pos[:,0], X_pos[:,1], label="Class +1", alpha=0.8)\nplt.scatter(X_neg[:,0], X_neg[:,1], label="Class -1", alpha=0.8)\nplt.xlabel("x₁")\nplt.ylabel("x₂")\nplt.title("Synthetic binary classification dataset")\nplt.legend()\nplt.grid(True, alpha=0.3)\nplt.show()

In [ ]:
def fit_classification_models(X, y):\n    log_model = make_pipeline(\n        StandardScaler(),\n        LogisticRegression(random_state=0)\n    )\n    log_model.fit(X, y)\n\n    svm_model = make_pipeline(\n        StandardScaler(),\n        LinearSVC(random_state=0, dual="auto")\n    )\n    svm_model.fit(X, y)\n\n    return {"logistic": log_model, "svm": svm_model}

In [ ]:
models = fit_classification_models(X_cls, y_cls)

In [ ]:
def plot_decision_boundary(ax, model, X, y, title):\n    x_min, x_max = X[:, 0].min() - 1.0, X[:, 0].max() + 1.0\n    y_min, y_max = X[:, 1].min() - 1.0, X[:, 1].max() + 1.0\n    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),\n                         np.linspace(y_min, y_max, 300))\n    grid = np.c_[xx.ravel(), yy.ravel()]\n\n    Z = model.decision_function(grid).reshape(xx.shape)\n\n    ax.contour(xx, yy, Z, levels=[0], linewidths=2)\n    ax.scatter(X[y==1, 0], X[y==1, 1], label="Class +1", alpha=0.8)\n    ax.scatter(X[y==-1, 0], X[y==-1, 1], label="Class -1", alpha=0.8)\n    ax.set_xlabel("x₁")\n    ax.set_ylabel("x₂")\n    ax.set_title(title)\n    ax.grid(True, alpha=0.3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))\n\nplot_decision_boundary(axes[0], models["logistic"], X_cls, y_cls, "Logistic regression")\nplot_decision_boundary(axes[1], models["svm"], X_cls, y_cls, "Linear SVM")\n\naxes[0].legend()\nplt.tight_layout()\nplt.show()

## Similar decision boundaries, different objectives\n\nIn practice, logistic regression and linear SVM often produce very similar decision boundaries.\n\nHowever, they are optimizing **different objectives**:\n\n- Logistic regression minimizes the logistic loss.\n- Linear SVM minimizes the hinge loss.

### Question\n\nIf two classifiers produce almost the same decision boundary, are they solving the same optimization problem?\n\nWhy or why not?